# Single-Protein DDPM Training

This notebook trains a DDPM model on **one specific protein** by learning to denoise different conformational states (frames) from an MD trajectory.

**Workflow:**
1. Specify protein name and parameters
2. Dynamically create/load trajectory data for that protein
3. Train DDPM to denoise frames
4. Generate and evaluate results

## Step 1: Environment Setup

In [ ]:
# Check environment
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

# Install dependencies
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio'], check=True)
    subprocess.run(['pip', 'install', '-q',
                    'omegaconf', 'pandas', 'tqdm', 'numpy', 'scipy', 'matplotlib',
                    'lightning', 'mdtraj', 'requests', 'tensorboard',
                    'optuna', 'optuna-integration', 'einops', 'py3Dmol'], check=True)
else:
    # Local: install packages that may not be pre-installed in every environment
    import subprocess
    subprocess.run(
        ['pip', 'install', '-q',
         'tensorboard', 'optuna', 'optuna-integration', 'einops', 'py3Dmol'],
        check=True,
    )

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [2]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Ensure the Drive data directory exists (symlink created after repo clone)
    os.makedirs('/content/drive/MyDrive/protein_data/data', exist_ok=True)
    print("Drive mounted")
else:
    # Running locally: just make sure data/ exists
    os.makedirs('data', exist_ok=True)
    print(f"Local mode; data dir: {os.path.abspath('data')}")

Local mode; data dir: /home/jiwonjjeong/repos/winter-gen-pproject/data


## Step 2: Get Code from GitHub

Clone your repository to get the `gen_model` code (no data needed)

In [3]:
import os, subprocess

REPO_URL = "https://github.com/JiwonJJeong/winter-gen-pproject.git"

if IN_COLAB:
    if not os.path.exists('winter-gen-pproject'):
        print(f"Cloning {REPO_URL} ...")
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('winter-gen-pproject')
    # Create data/ symlink inside repo root (after chdir) so all cells find it
    if not os.path.islink('data'):
        os.symlink('/content/drive/MyDrive/protein_data/data', 'data')
        print("data/ -> /content/drive/MyDrive/protein_data/data")
    # Pull latest changes and clear bytecode cache
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    for cmd in [
        'find gen_model -name "*.pyc" -delete',
        'find gen_model -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null; true',
    ]:
        subprocess.run(cmd, shell=True)
    print("Code updated")
else:
    # Already inside the repo; just verify gen_model/ is present
    if not os.path.isdir('gen_model'):
        raise RuntimeError(
            "gen_model/ not found. Run this notebook from the repository root:\n"
            "  cd /path/to/winter-gen-pproject && jupyter notebook"
        )
    print(f"Local repo root: {os.path.abspath('.')}")

import subprocess as _sp
result = _sp.run(['ls', 'gen_model/'], capture_output=True, text=True)
print(result.stdout)
print("Code ready")

Local repo root: /home/jiwonjjeong/repos/winter-gen-pproject
README.md
__init__.py
__pycache__
all_atom.py
dataset.py
diffusion
ema.py
geometry.py
inference_se3.py.bak
models
parsing.py
residue_constants.py
rigid_utils.py
simple_inference.py
simple_train.py
split.py
splits
tensor_utils.py
test_dataset.py
train_se3.py.bak
utils.py

Code ready


## Step 3: Configure Protein and Training

**Customize these settings for your protein:**

In [ ]:
from omegaconf import OmegaConf

# ── HPO / LoRA settings ───────────────────────────────────────────────────────
HPO_CONFIG = dict(
    lora_r             = 0,      # HPO always uses full fine-tuning (r=0 dominated all LoRA ranks)
    lora_alpha         = 0.0,    # 0 → auto-set to 2×lora_r
    n_trials           = 5,      # Optuna trials to run during HPO
    epochs_per_trial   = 5,      # Short epochs per trial (keep small for speed)
    pruner             = 'asha', # 'none' or 'asha' (Asynchronous Successive Halving)
    local_attn_sigma   = 15.0,   # IPA spatial cutoff (Å); ~2% attn beyond this distance
    seq_tfmr_sigma     = 20.0,   # seq-transformer cutoff (residues)
)

# ── Pre-computed HPO results ──────────────────────────────────────────────────
# Set RUN_HPO = False (default) to skip the Optuna search and use the results
# below directly.  Set RUN_HPO = True to run a fresh search.
# Note: lr is NOT stored here — it is found automatically by the LR finder in
# Step 9, which is more reliable than short-trial HPO for learning rate search.
RUN_HPO = False

BEST_PARAMS_PRESET = {
    'seq_tfmr_num_layers': 1,
    'lora_r':              0,
    'lora_alpha':          0.0,
    # best val_loss = 3.601775  (trial #0, 5 trials × 5 epochs)
}

# ── Protein / training settings ───────────────────────────────────────────────
protein_config = OmegaConf.create({
    'protein': {
        'name':              '4o66_C',
        'replica':           1,
        'train_early_ratio': 0.3,
        'train_ratio':       0.4,
        'val_ratio':         0.2,
    },
    'training': {
        'batch_size':        8,
        'num_epochs':        100,
        'learning_rate':     1e-4,
        'rot_loss_weight':   1.0,
        'trans_loss_weight': 1.0,
        'psi_loss_weight':   1.0,
    },
    'inference': {
        'num_samples': 5,
        'num_steps':   200,
    },
})
prot_cfg = protein_config.protein

print("Configuration Summary:")
print("="*70)
print(f"Protein   : {prot_cfg.name}_R{prot_cfg.replica}")
print(f"LoRA      : r={HPO_CONFIG['lora_r']}  (0 = full fine-tuning)")
print(f"HPO       : {'RUN search' if RUN_HPO else 'use preset'} | {HPO_CONFIG['n_trials']} trials × {HPO_CONFIG['epochs_per_trial']} epochs | pruner={HPO_CONFIG['pruner']}")
print(f"Training  : {protein_config.training.num_epochs} epochs | batch={protein_config.training.batch_size} | lr=auto (LR finder)")
print("="*70)

IGSO3_CACHE = '/content/drive/MyDrive/protein_data/igso3_cache' if IN_COLAB else '/tmp/igso3_cache'


## Step 4: Create/Load Protein Data

This creates the data structure for your specified protein

In [5]:
import os

# Guard: data/ must be symlinked to Drive before downloading.
# If data/ is not a symlink, it means the Drive mount cell (Step 2) was not run
# first and any download would land in the ephemeral VM filesystem (lost on restart).
if IN_COLAB and not os.path.islink('data'):
    raise RuntimeError(
        "data/ is not linked to Google Drive.\n"
        "Run the 'Mount Google Drive' cell above first, then re-run this cell."
    )

# Use the automatic download and prep script
prot_name = protein_config.protein.name
!python scripts/download_and_prep.py {prot_name} --data_dir ./data --out_dir ./data --suffix _latent

# Setup paths for verification
prot_cfg = protein_config.protein
PROTEIN_FULL_NAME = f"{prot_cfg.name}_R{prot_cfg.replica}"
protein_dir = f'data/{prot_cfg.name}'
traj_path = f'{protein_dir}/{PROTEIN_FULL_NAME}_latent.npy'

if os.path.exists(traj_path):
    print(f"\u2705 Data ready at: {traj_path}")
else:
    print(f"\u274c Error: Data not found at {traj_path}")


Data for 4o66_C seems to exist at ./data/4o66_C. Skipping download.
Detected timestep: 10.0 ps
Preprocessing output ./data/4o66_C/4o66_C_R1_latent.npy already exists. Skipping.
Preprocessing output ./data/4o66_C/4o66_C_R2_latent.npy already exists. Skipping.
Preprocessing output ./data/4o66_C/4o66_C_R3_latent.npy already exists. Skipping.
Creating 4-way split (Early: 5.0ns, Ratios: [0.6, 0.2, 0.2])...
Found 3 trajectory file(s) for 4o66_C.
  4o66_C_R1_latent: Total 10001, Timestep 10.0ps
    Early: [0:500], Train: [500:6200], Val: [6200:8100], Test: [8100:10001]
  4o66_C_R2_latent: Total 10001, Timestep 10.0ps
    Early: [0:500], Train: [500:6200], Val: [6200:8100], Test: [8100:10001]
  4o66_C_R3_latent: Total 10001, Timestep 10.0ps
    Early: [0:500], Train: [500:6200], Val: [6200:8100], Test: [8100:10001]
Updated splits saved to gen_model/splits/frame_splits.csv
✅ Data ready at: data/4o66_C/4o66_C_R1_latent.npy


In [ ]:
# Reference Trajectory Visualization
# Load a sample of MD frames and show them inline before training.
import numpy as np, pandas as pd, os
from gen_model.utils.structure_utils import (
    save_ca_trajectory_pdb, show_py3dmol_trajectory,
)

# Load residue sequence from atlas CSV
_seq_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
seqres  = _seq_df.loc[prot_cfg.name, 'seqres']
print(f"Protein : {prot_cfg.name}  ({len(seqres)} residues)")

# Sample 30 evenly-spaced Cα frames from the raw MD trajectory
N_REF  = 30
_arr   = np.lib.format.open_memmap(traj_path, 'r')
_idxs  = np.linspace(0, len(_arr) - 1, N_REF, dtype=int)
ref_ca = _arr[_idxs, :, 1, :].astype(np.float32)          # [N_REF, N, 3]
ref_ca = ref_ca - ref_ca.mean(axis=(0, 1), keepdims=True)  # centre at origin

os.makedirs('outputs', exist_ok=True)
ref_pdb = f'outputs/{prot_cfg.name}_reference.pdb'
save_ca_trajectory_pdb(ref_ca, ref_pdb, seqres)
print(f"Reference PDB saved : {ref_pdb}  ({N_REF} frames)")

try:
    view = show_py3dmol_trajectory(ref_ca, seqres, color='blue', interval=150)
    print(f"\n\u25b6 C\u03b1 trace: {N_REF} reference MD frames (blue), animated:")
    view.show()
except ImportError:
    print("py3Dmol not found \u2014 install with:  pip install py3Dmol")
    print(f"Or open {ref_pdb} in PyMOL / VMD / UCSF ChimeraX")


## Step 5: Configure Dataset and Model

In [ ]:
import sys
sys.path.insert(0, '.')

from gen_model.train_unconditional    import SE3Module
from gen_model.train_base             import default_se3_conf, default_model_conf
from gen_model.data.dataset           import MDGenDataset
from gen_model.diffusion.se3_diffuser import SE3Diffuser

print("✓ Modules imported")

data_config = OmegaConf.create({
    'data_dir':       'data',
    'atlas_csv':      'gen_model/splits/atlas.csv',
    'train_split':    'gen_model/splits/frame_splits.csv',
    'suffix':         '_latent',
    'frame_interval': None,
    'crop_ratio':     0.95,
    'min_t':          0.01,
    'pep_name':       prot_cfg.name,    # single-protein filter
    'replica':        prot_cfg.replica, # single-replica filter
})

print(f"\nDataset config:")
print(f"  Protein : {data_config.pep_name}_R{data_config.replica}")

## Step 6: Load Dataset

In [ ]:
# Build SE3 diffuser from canonical defaults (schedule_gamma=1.0, no coordinate_scaling)
print("Initialising SE3Diffuser...")
se3_conf     = default_se3_conf(cache_dir=IGSO3_CACHE)
se3_diffuser = SE3Diffuser(se3_conf)
print("✓ SE3Diffuser ready\n")

# MDGenDataset computes coord_scale dynamically from training data (~1/std of Ca coords)
print("Loading datasets...")
train_dataset = MDGenDataset(
    args=data_config, diffuser=se3_diffuser, mode='train',
    repeat=1, num_consecutive=1, stride=1,
)
val_dataset = MDGenDataset(
    args=data_config, diffuser=se3_diffuser, mode='val',
    repeat=1, num_consecutive=1, stride=1,
)
val_dataset.coord_scale = float(train_dataset.coord_scale)

print(f"\n✓ Training frames  : {len(train_dataset)}")
print(f"✓ Validation frames: {len(val_dataset)}")
print(f"✓ coord_scale      : {train_dataset.coord_scale:.4f}  (computed from data)")

sample     = train_dataset[0]
n_residues = sample['res_mask'].shape[0]
print(f"\nSample keys   : {list(sample.keys())}")
print(f"Residues      : {n_residues}")
print(f"rigids_t shape: {sample['rigids_t'].shape}  (per-residue SE3 frame at time t)")
print(f"t (noise)     : {sample['t']:.4f}")

## Step 7: Preview Model Architecture

Builds a preview model with default params to show the architecture. HPO (Step 8) will find the optimal hyperparameters; the full training (Step 9) rebuilds the model with those best params.

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}\n")

# Build model config with optional LoRA adapters
model_conf = default_model_conf(
    lora_r     = HPO_CONFIG['lora_r'],
    lora_alpha = HPO_CONFIG['lora_alpha'],
)

model_pl = SE3Module(
    model_conf        = model_conf,
    se3_conf          = se3_conf,
    lr                = protein_config.training.learning_rate,
    rot_loss_weight   = protein_config.training.rot_loss_weight,
    trans_loss_weight = protein_config.training.trans_loss_weight,
    psi_loss_weight   = protein_config.training.psi_loss_weight,
)

total     = sum(p.numel() for p in model_pl.parameters())
trainable = sum(p.numel() for p in model_pl.parameters() if p.requires_grad)
print(f"✓ Score Network: {total:,} total params | {trainable:,} trainable ({100*trainable/total:.1f}%)")
print(f"  LoRA r={HPO_CONFIG['lora_r']} → {'LoRA adapters only' if HPO_CONFIG['lora_r'] > 0 else 'full fine-tuning'}")

## Step 8: Hyperparameter Optimisation (HPO)

Runs short Optuna trials (`epochs_per_trial` each) on this specific protein to search for the best combination of learning rate, LoRA rank, loss weights, and noise schedule curvature.
The `best_params` dict is automatically passed to the full training step.

In [ ]:
# =============================================================================
# Step 8: Hyperparameter Optimisation (HPO)
# RUN_HPO = False  → load BEST_PARAMS_PRESET from the config cell (fast, default)
# RUN_HPO = True   → run a fresh Optuna search and overwrite best_params
# =============================================================================
import types, os, optuna
from torch.utils.data import DataLoader
from gen_model.hpo import _suggest_hparams, _build_se3_conf, _build_pruner
from gen_model.train_base import default_model_conf
from gen_model.diffusion.se3_diffuser import SE3Diffuser

if not RUN_HPO:
    best_params = dict(BEST_PARAMS_PRESET)
    print("HPO skipped — using pre-computed best params (set RUN_HPO=True to re-run search):")
    for _k, _v in best_params.items():
        print(f"  {_k:25s}: {_v}")
else:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    HPO_SAVE_DIR = f'checkpoints/hpo/{prot_cfg.name}_cond'
    os.makedirs(HPO_SAVE_DIR, exist_ok=True)

    def _hpo_objective(trial):
        import torch, lightning as L
        from lightning.pytorch.callbacks import ModelCheckpoint
        from optuna.integration.pytorch_lightning import PyTorchLightningPruningCallback
        from gen_model.train_conditional import ConditionalSE3Module as _Module
        from gen_model.data.dataset import ConditionalMDGenDataset as _DS

        hp         = _suggest_hparams(trial)
        _se3_conf  = _build_se3_conf(hp, cache_dir=IGSO3_CACHE)
        _mc        = default_model_conf(
            use_temporal_embedding=True,
            lora_r=hp['lora_r'], lora_alpha=hp['lora_alpha'],
            local_attn_sigma=HPO_CONFIG['local_attn_sigma'],
            seq_tfmr_sigma=HPO_CONFIG['seq_tfmr_sigma'],
            seq_tfmr_num_layers=hp['seq_tfmr_num_layers'],
        )
        _diffuser  = SE3Diffuser(_se3_conf)
        _train = _DS(args=data_config, diffuser=_diffuser, mode="train", repeat=1, num_consecutive=1, stride=1)
        _val   = _DS(args=data_config, diffuser=_diffuser, mode="val",   repeat=1, num_consecutive=1, stride=1)
        _val.coord_scale = float(_train.coord_scale)

        _model = _Module(model_conf=_mc, se3_conf=_se3_conf, lr=hp['lr'],
                         rot_loss_weight=1.0, trans_loss_weight=1.0, psi_loss_weight=0.5)

        _pruning_cb = PyTorchLightningPruningCallback(trial, monitor='val_loss')
        _ckpt_cb    = ModelCheckpoint(
            dirpath=os.path.join(HPO_SAVE_DIR, f'trial_{trial.number}'),
            filename='best', monitor='val_loss', save_top_k=1, mode='min',
        )
        _nw = 0
        _pm = torch.cuda.is_available()
        _trainer = L.Trainer(
            max_epochs=HPO_CONFIG['epochs_per_trial'], accelerator='auto', devices='auto',
            callbacks=[_pruning_cb, _ckpt_cb],
            precision='16-mixed' if _pm else 32,
            enable_progress_bar=False, logger=False, enable_model_summary=False,
            limit_val_batches=0.5,
        )
        try:
            _trainer.fit(_model,
                train_dataloaders=DataLoader(_train, batch_size=protein_config.training.batch_size,
                    shuffle=True, num_workers=_nw, pin_memory=_pm, persistent_workers=_nw > 0),
                val_dataloaders=DataLoader(_val, batch_size=protein_config.training.batch_size,
                    shuffle=False, num_workers=_nw, pin_memory=_pm, persistent_workers=_nw > 0))
        except optuna.exceptions.TrialPruned:
            raise
        return _trainer.callback_metrics['val_loss'].item()

    _pruner = _build_pruner(types.SimpleNamespace(
        pruner=HPO_CONFIG['pruner'], asha_min_resource=1, asha_reduction_factor=3))
    study = optuna.create_study(
        direction='minimize',
        storage=f'sqlite:///{HPO_SAVE_DIR}/optuna.db',
        study_name=f'se3_{prot_cfg.name}_cond',
        load_if_exists=True, pruner=_pruner,
    )

    def _trial_callback(study, trial):
        status = 'pruned' if trial.value is None else f'val_loss={trial.value:.6f}'
        best   = f'{study.best_value:.6f}' if study.best_trial else 'n/a'
        print(f"  Trial {trial.number + 1:2d}/{HPO_CONFIG['n_trials']}  "
              f"{status}  |  best so far: {best}")

    print(f"Running {HPO_CONFIG['n_trials']} HPO trials ({HPO_CONFIG['epochs_per_trial']} epochs each)  protein={prot_cfg.name}...")
    study.optimize(_hpo_objective, n_trials=HPO_CONFIG['n_trials'],
                   catch=(Exception,), callbacks=[_trial_callback])

    _bt = study.best_trial
    best_params = dict(_bt.params)
    best_params['lora_alpha'] = float(2 * best_params.get('lora_r', 0))

    print(f"\n{'='*62}")
    print(f"HPO complete  {len(study.trials)} trials | best val_loss = {_bt.value:.6f}  (trial #{_bt.number})")
    print(f"{'='*62}")
    print("Best params (passed automatically to full training in Step 9):")
    for _k, _v in best_params.items():
        print(f"  {_k:25s}: {_v}")


## Step 9: Full Training (HPO Best Params)

In [ ]:
import lightning as L

class WeightHistogramCallback(L.Callback):
    """Log per-parameter weight and gradient histograms to TensorBoard each epoch.

    After training, open TensorBoard and navigate to:
      Histograms   — weight distributions for every named layer, over epochs
      Distributions — alternative time-series view of the same data

    Gradients are logged only when they exist (i.e. after at least one backward pass).
    Large models may generate many histogram series — filter by layer name in TensorBoard.
    """

    def on_train_epoch_end(self, trainer, pl_module):
        # Support both single logger and list-of-loggers
        loggers = trainer.loggers if hasattr(trainer, 'loggers') else [trainer.logger]
        tb = None
        for lg in loggers:
            if hasattr(lg, 'experiment') and hasattr(lg.experiment, 'add_histogram'):
                tb = lg.experiment
                break
        if tb is None:
            return

        epoch = trainer.current_epoch
        for name, param in pl_module.named_parameters():
            if not param.requires_grad:
                continue
            tb.add_histogram(f'weights/{name}', param.detach().cpu(), epoch)
            if param.grad is not None:
                tb.add_histogram(f'grads/{name}', param.grad.detach().cpu(), epoch)

print("\u2713 WeightHistogramCallback defined")

In [ ]:
# ── TensorBoard inline viewer ─────────────────────────────────────────────────
# Tabs to look at:
#   Scalars      → train_loss_step / val_loss curves over training
#   Histograms   → per-parameter weight and gradient distributions, per epoch
#   Distributions → same data as Histograms in a violin/area view
#
# Can be run before training finishes — TensorBoard reads logs incrementally.
# For a standalone browser tab instead:  tensorboard --logdir tb_logs/
# ─────────────────────────────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir tb_logs/

In [ ]:
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers   import TensorBoardLogger, CSVLogger
from lightning.pytorch.tuner     import Tuner
from torch.utils.data import DataLoader, ConcatDataset
import numpy as np, os
from gen_model.hpo        import _build_se3_conf
from gen_model.train_base import default_model_conf, default_se3_conf
from gen_model.diffusion.se3_diffuser import SE3Diffuser
from gen_model.train_conditional import ConditionalSE3Module
from gen_model.data.dataset import ConditionalMDGenDataset

# ── Hyperparameters: use HPO best if available, else fall back to defaults ────
if 'best_params' in vars() and best_params:
    print("\u2713 Using HPO best hyperparameters for full training:")
    _seq_tfmr_num_layers = best_params.get('seq_tfmr_num_layers', 1)
    _se3_conf = _build_se3_conf(best_params, cache_dir=IGSO3_CACHE)
else:
    print("HPO not run \u2014 using defaults from configuration cell:")
    _seq_tfmr_num_layers = BEST_PARAMS_PRESET['seq_tfmr_num_layers']
    _se3_conf = default_se3_conf(cache_dir=IGSO3_CACHE)

_lora_r, _lora_alpha = 0, 0.0
_rot_w, _trans_w, _psi_w = 1.0, 1.0, 0.5
_local_attn_sigma = HPO_CONFIG['local_attn_sigma']
_seq_tfmr_sigma   = HPO_CONFIG['seq_tfmr_sigma']
print(f"  seq_tfmr_num_layers={_seq_tfmr_num_layers}  rot_w={_rot_w:.2f}  trans_w={_trans_w:.2f}  psi_w={_psi_w:.2f}  lr=auto")

# ── Rebuild datasets ──────────────────────────────────────────────────────────
print("\nRebuilding datasets with chosen se3_conf...")
_diffuser = SE3Diffuser(_se3_conf)
_train_ds = ConditionalMDGenDataset(args=data_config, diffuser=_diffuser, mode="train",
                                    repeat=1, num_consecutive=1, stride=1,
                                    max_k=3, current_max_k=1)
_val_ds   = ConditionalMDGenDataset(args=data_config, diffuser=_diffuser, mode="val",
                                    repeat=1, num_consecutive=1, stride=1,
                                    max_k=3, current_max_k=1)
_val_ds.coord_scale = float(_train_ds.coord_scale)
_combined_ds = ConcatDataset([_train_ds, _val_ds])
train_loader = DataLoader(_combined_ds, batch_size=protein_config.training.batch_size, shuffle=True)
val_loader   = DataLoader(_val_ds,   batch_size=protein_config.training.batch_size, shuffle=False)
print(f"\u2713 Train: {len(_combined_ds)} frames (train={len(_train_ds)} + val={len(_val_ds)}) | Val monitor: {len(_val_ds)} frames")

# ── Save coord_scale ──────────────────────────────────────────────────────────
_ckpt_save_dir = f'checkpoints/{prot_cfg.name}_cond_se3'
os.makedirs(_ckpt_save_dir, exist_ok=True)
_coord_scale_path = os.path.join(_ckpt_save_dir, 'coord_scale.npy')
np.save(_coord_scale_path, np.array([_train_ds.coord_scale]))
print(f"\u2713 coord_scale={_train_ds.coord_scale:.4f} saved \u2192 {_coord_scale_path}")

# ── Build model ───────────────────────────────────────────────────────────────
_mc = default_model_conf(use_temporal_embedding=True, lora_r=_lora_r, lora_alpha=_lora_alpha,
                         local_attn_sigma=_local_attn_sigma, seq_tfmr_sigma=_seq_tfmr_sigma,
                         seq_tfmr_num_layers=_seq_tfmr_num_layers)
model_pl = ConditionalSE3Module(
    model_conf=_mc, se3_conf=_se3_conf, lr=1e-3,  # placeholder; LR finder sets real value
    rot_loss_weight=_rot_w, trans_loss_weight=_trans_w, psi_loss_weight=_psi_w,
)
_tot = sum(p.numel() for p in model_pl.parameters())
_trn = sum(p.numel() for p in model_pl.parameters() if p.requires_grad)
print(f"\u2713 Model: {_tot:,} params | {_trn:,} trainable ({100*_trn/_tot:.1f}%)")

# ── LR Finder ─────────────────────────────────────────────────────────────────
# Sweeps lr exponentially and finds the steepest loss descent point.
# Falls back to 3e-4 if the finder diverges (common with diffusion score losses).
print("\nRunning LR finder (200 steps)...")
_finder_trainer = L.Trainer(
    accelerator="auto", devices=1,
    precision="16-mixed" if __import__('torch').cuda.is_available() else 32,
    enable_progress_bar=False, logger=False, enable_model_summary=False,
)
_tuner     = Tuner(_finder_trainer)
_lr_finder = _tuner.lr_find(
    model_pl, train_dataloaders=train_loader, val_dataloaders=val_loader,
    min_lr=1e-6, max_lr=1e-3, num_training=200, mode='exponential',
)
_lr = _lr_finder.suggestion()
if _lr is None:
    _lr = 3e-4
    print(f"⚠ LR finder could not converge — using fallback lr={_lr:.2e}")
else:
    print(f"✓ LR finder suggestion: {_lr:.2e}")
    _lr_finder.plot(suggest=True)
model_pl.lr = _lr

# ── Loggers ───────────────────────────────────────────────────────────────────
_tb_logger  = TensorBoardLogger('tb_logs', name=f'{prot_cfg.name}_cond')
_csv_logger = CSVLogger('lightning_logs')
print(f"\nTensorBoard logs : tb_logs/{prot_cfg.name}_cond/version_*/")

# ── Callbacks ─────────────────────────────────────────────────────────────────
_ckpt_cb = ModelCheckpoint(
    dirpath=_ckpt_save_dir,
    filename='cond-se3-{epoch:02d}-{val_loss:.4f}',
    save_top_k=3, monitor='val_loss', mode='min', save_last=True,
)
_early_stop_cb = EarlyStopping(
    monitor='val_loss', patience=10, mode='min', verbose=True,
)

# ── Train ─────────────────────────────────────────────────────────────────────
trainer = L.Trainer(
    max_epochs=protein_config.training.num_epochs,
    accelerator="auto", devices=1,
    logger=[_csv_logger, _tb_logger],
    callbacks=[_ckpt_cb, WeightHistogramCallback(), _early_stop_cb],
    precision="16-mixed" if __import__('torch').cuda.is_available() else 32,
)
trainer.fit(model_pl, train_dataloaders=train_loader, val_dataloaders=val_loader)


In [ ]:
import glob

ckpt_dir  = f'checkpoints/{prot_cfg.name}_cond_se3'
ckpt_list = sorted(glob.glob(f'{ckpt_dir}/*.ckpt'))

if ckpt_list:
    print(f"Checkpoints in {ckpt_dir}/:")
    for p in ckpt_list:
        print(f"  {p}")
    # Prefer non-last checkpoints (ranked by val_loss in filename)
    ranked = [p for p in ckpt_list if 'last' not in p]
    best_ckpt = ranked[-1] if ranked else ckpt_list[-1]
    print(f"
Best checkpoint : {best_ckpt}")
else:
    best_ckpt = None
    print(f"No checkpoints found in {ckpt_dir}/")
    print("Run Step 9 (training) first.")


## Step 10: Evaluate

In [ ]:
# =============================================================================
# Inline Inference + Visualization  (Conditional SE(3))
# Loads the trained ConditionalSE3Module, picks a source frame, runs the
# reverse SE(3) SDE conditioned on it, and shows the predicted next frame.
# =============================================================================
import glob, torch, os, numpy as np
from gen_model.train_base              import default_se3_conf, default_model_conf
from gen_model.train_conditional       import ConditionalSE3Module
from gen_model.diffusion.se3_diffuser  import SE3Diffuser
from gen_model.inference_conditional   import run_reverse_sde, load_source_frame, compute_coord_scale
from gen_model.data.residue_constants  import restype_order
from gen_model.utils.structure_utils   import (
    aatype_to_seqres, rigids_to_ca_trajectory,
    save_ca_trajectory_pdb, show_py3dmol_trajectory,
)

_device   = 'cuda' if torch.cuda.is_available() else 'cpu'
_ckpt_dir = f'checkpoints/{prot_cfg.name}_cond_se3'
_ckpts    = [p for p in sorted(glob.glob(f'{_ckpt_dir}/*.ckpt')) if 'last' not in p]
_ckpt     = _ckpts[-1] if _ckpts else None

# ── Inference settings ───────────────────────────────────────────────────────
SOURCE_FRAME_IDX = 1000   # source frame index in the trajectory
K                = 1      # temporal gap: 1 = predict next frame
N_INLINE_SAMPLES = 3      # independent samples to generate
N_INLINE_STEPS   = 100    # reverse-SDE steps

if _ckpt is None:
    print(f"No checkpoint found in {_ckpt_dir} — train the model first (Step 9).")
    gen_ca = None
else:
    print(f"Checkpoint   : {_ckpt}")

    # ── seq_tfmr_num_layers must match the checkpoint ─────────────────────────
    _seq_tfmr_num_layers = (
        best_params.get('seq_tfmr_num_layers', BEST_PARAMS_PRESET['seq_tfmr_num_layers'])
        if 'best_params' in vars() and best_params
        else BEST_PARAMS_PRESET['seq_tfmr_num_layers']
    )

    # ── Load model ────────────────────────────────────────────────────────────
    _se3_conf   = default_se3_conf(cache_dir=IGSO3_CACHE)
    _model_conf = default_model_conf(
        use_temporal_embedding=True,
        lora_r=HPO_CONFIG['lora_r'],
        lora_alpha=HPO_CONFIG['lora_alpha'],
        seq_tfmr_num_layers=_seq_tfmr_num_layers,
    )
    _module = ConditionalSE3Module.load_from_checkpoint(
        _ckpt, model_conf=_model_conf, se3_conf=_se3_conf, map_location=_device,
    ).to(_device).eval()
    _diffuser = SE3Diffuser(_se3_conf)
    print(f"Model loaded  : seq_tfmr_num_layers={_seq_tfmr_num_layers}")

    # ── coord_scale ───────────────────────────────────────────────────────────
    _coord_scale_path = os.path.join(_ckpt_dir, 'coord_scale.npy')
    if os.path.exists(_coord_scale_path):
        _coord_scale = float(np.load(_coord_scale_path)[0])
        print(f"coord_scale  : {_coord_scale:.4f}  (loaded from checkpoint dir)")
    else:
        _coord_scale = compute_coord_scale(traj_path)
        print(f"coord_scale  : {_coord_scale:.4f}  (recomputed — run Step 9 first for exact match)")

    # ── Load source frame ─────────────────────────────────────────────────────
    _, _seqres, _ca_pos, _, _aatype = load_source_frame(
        traj_path, SOURCE_FRAME_IDX,
        atlas_csv='gen_model/splits/atlas.csv',
        protein_name=prot_cfg.name,
    )
    _centroid  = _ca_pos.mean(axis=0, keepdims=True)                              # [1, 3]
    _sc_ca_t   = torch.from_numpy((_ca_pos - _centroid) * _coord_scale).float()  # [N, 3]
    _res_mask  = torch.ones(len(_seqres), dtype=torch.float32)
    print(f"Source frame : [{SOURCE_FRAME_IDX}]  ({len(_seqres)} residues, predicting k={K})")

    # ── Generate samples ──────────────────────────────────────────────────────
    _rigids = []
    for _s in range(N_INLINE_SAMPLES):
        _rigid = run_reverse_sde(
            model=_module, diffuser=_diffuser,
            sc_ca_t=_sc_ca_t, aatype=_aatype, res_mask=_res_mask,
            k=K, num_steps=N_INLINE_STEPS, device=_device,
        )
        _rigids.append(_rigid)
        print(f"  Sample {_s + 1}/{N_INLINE_SAMPLES} done")

    gen_ca = rigids_to_ca_trajectory(_rigids, _coord_scale)  # [N_SAMPLES, N, 3]
    _seqres_for_viz = aatype_to_seqres(_aatype)

    gen_pdb = f'outputs/{prot_cfg.name}_cond_generated.pdb'
    os.makedirs('outputs', exist_ok=True)
    save_ca_trajectory_pdb(gen_ca, gen_pdb, _seqres_for_viz)
    print(f"\nGenerated    : {gen_ca.shape}  (samples × residues × xyz, Å)")
    print(f"Saved PDB    : {gen_pdb}")

    # ── Quick RMSD sanity check ───────────────────────────────────────────────
    _rmsd = float(np.sqrt(np.mean((gen_ca - _ca_pos[None]) ** 2)))
    print(f"Mean RMSD to source frame : {_rmsd:.2f} Å  (expected ~several Å for k={K})")

    try:
        _seqres_str = _seqres if isinstance(_seqres, str) else ''.join(_seqres)

        # Source frame (blue)
        _src_ca_ang = _ca_pos[None]  # [1, N, 3]
        view_src = show_py3dmol_trajectory(_src_ca_ang, _seqres_str, color='blue', interval=500)
        print("\n▶ Source frame (blue):")
        view_src.show()

        # Generated conformations (orange)
        view_gen = show_py3dmol_trajectory(gen_ca, _seqres_str, color='orange', interval=300)
        print("\n▶ Generated frames (orange, animated):")
        view_gen.show()

    except ImportError:
        print("\npy3Dmol not installed — install with:  pip install py3Dmol")
        print(f"Or open {gen_pdb} in PyMOL / VMD / UCSF ChimeraX")


In [ ]:
# Inference runs the reverse SE(3) SDE from pure noise to generate new conformations.
# Run from terminal after training completes:

npy_path = f'data/{prot_cfg.name}/{prot_cfg.name}_R{prot_cfg.replica}_latent.npy'

if best_ckpt:
    print("To generate samples, run from terminal:")
    print()
    print(f"  python gen_model/inference_unconditional.py \\")
    print(f"      --checkpoint {best_ckpt} \\")
    print(f"      --npy_path {npy_path} \\")
    print(f"      --protein_name {prot_cfg.name} \\")
    print(f"      --num_steps {protein_config.inference.num_steps} \\")
    print(f"      --num_samples {protein_config.inference.num_samples} \\")
    print(f"      --out_dir outputs/{prot_cfg.name}_se3")
else:
    print("Train a model first (Step 8), then run inference.")


## Step 10: Generate Samples

In [ ]:
import matplotlib.pyplot as plt, pandas as pd, glob

# Lightning's CSVLogger writes metrics to lightning_logs/version_N/metrics.csv
log_files = sorted(glob.glob('lightning_logs/version_*/metrics.csv'))

if log_files:
    df  = pd.read_csv(log_files[-1])
    val = df[['epoch', 'val_loss']].dropna()
    trn = df[['step',  'train_loss_step']].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    if not val.empty:
        axes[0].plot(val['epoch'], val['val_loss'], 'o-', color='steelblue')
        axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Loss')
        axes[0].set_title(f'Validation Loss — {prot_cfg.name}'); axes[0].grid(alpha=0.3)
    if not trn.empty:
        axes[1].plot(trn['step'], trn['train_loss_step'], alpha=0.6, color='steelblue')
        axes[1].set_xlabel('Step'); axes[1].set_ylabel('Train Loss (step)')
        axes[1].set_title('Training Loss'); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"Log: {log_files[-1]}")
else:
    print("No training logs found. Run Step 8 (training) first.")
    print("Expected location: lightning_logs/version_*/metrics.csv")

## Step 11: Visualize

In [ ]:
import os, subprocess

save_zip = f'{prot_cfg.name}_results.zip'
dirs_to_pack = [d for d in [ckpt_dir, f'outputs/{prot_cfg.name}_se3'] if os.path.isdir(d)]

if dirs_to_pack:
    subprocess.run(['zip', '-rq', save_zip] + dirs_to_pack, check=True)
    print(f"✓ Packaged: {save_zip}  ({', '.join(dirs_to_pack)})")
    if IN_COLAB:
        from google.colab import files
        files.download(save_zip)
        print("✓ Download started")
    else:
        print(f"✓ Saved locally as {save_zip}")
else:
    print("Nothing to package yet — run Steps 8 and 10 first.")

## Step 12: Download Results

In [ ]:
# Package results
!zip -rq {prot_cfg.name}_results.zip {save_dir} {output_dir}

print(f"\nResults packaged: {prot_cfg.name}_results.zip")
print(f"  Checkpoints: {save_dir}/")
print(f"  Outputs: {output_dir}/")

if IN_COLAB:
    from google.colab import files
    files.download(f'{prot_cfg.name}_results.zip')
    print("\n✓ Download started")
else:
    print(f"\n✓ Saved locally as {prot_cfg.name}_results.zip")

## Summary

**What we did:**
1. ✓ Configured protein: `{protein_config.protein.name}_R{protein_config.protein.replica}`
2. ✓ Downloaded and preprocessed trajectory data using 
download_and_prep.py
3. ✓ Trained DDPM for {protein_config.training.num_epochs} epochs
4. ✓ Evaluated denoising on validation frames
5. ✓ Generated new conformations from noise

**Key insight:** The model learned the conformational space of **one specific protein** by training on different frames from its MD trajectory.

**To train on a different protein:**
- Change `protein_config.protein.name`
- Rerun from Step 3 onwards